<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/transformers-llm/07_rag_ve_hizalama_rlhf_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG ile Halüsinasyonu Azaltmak ve RLHF / DPO ile Model Hizalama

**Bölüm A – RAG (Retrieval-Augmented Generation):** Modelin bilmediği (kurum içi, güncel) bilgiyi sorulara cevap verirken bir bilgi tabanından **getirip** bağlama koyuyoruz. Aynı soruları RAG'li ve RAG'siz sorup halüsinasyon farkını görüyoruz.

**Bölüm B – Hizalama (Alignment):** Bir dil modelinin insan tercihlerine göre davranmasını sağlayan iki yaklaşımı matematiğiyle birlikte küçük bir oyuncak problemde uyguluyoruz:
- **RLHF**: Ödül modeli (Bradley–Terry) + KL kısıtlı politika optimizasyonu
- **DPO**: Ödül modeli olmadan, tercih çiftlerinden doğrudan optimizasyon

> Bölüm A'da küçük bir LLM (`Qwen2.5-0.5B-Instruct`, ~1 GB) indirilir. Colab'da GPU ile birkaç saniyede, CPU ile de çalışır.

In [ ]:
import re
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

---
# Bölüm A — RAG

### Neden halüsinasyon?

Bir LLM bilgiyi **parametrelerinde** saklar: eğitim verisinde görmediği, eğitimden sonra değişen veya kuruma özel olan bilgiyi bilemez. Bilmediği bir şey sorulunca da çoğu zaman "bilmiyorum" demek yerine **akıcı ama uydurma** bir cevap üretir.

### RAG akışı

```
 Soru ──► [Retriever] ──► en alakalı k parça ──► [İstem: talimat + bağlam + soru] ──► [LLM] ──► kaynaklı cevap
              ▲
     Bilgi tabanı (dokümanlar → parçalara bölünmüş → indekslenmiş)
```

1. **İndeksleme:** Dokümanlar parçalara (chunk) bölünür, her parça vektöre çevrilir.
2. **Getirme (retrieval):** Soru da vektöre çevrilir, en benzer $k$ parça bulunur.
3. **Zenginleştirme (augmentation):** Parçalar istemin içine bağlam olarak konur.
4. **Üretim (generation):** Model *sadece bağlamı kullanarak* cevap verir ve kaynak gösterir.

### Bilgi tabanı

Modelin kesinlikle bilemeyeceği **kurgusal** bir şirketin çalışan el kitabı:

In [ ]:
bilgi_tabani = {
    "izin-01": "Kuzey Yıldızı Yazılım'da tam zamanlı çalışanların yıllık izin hakkı ilk üç yıl için 18 iş günüdür. "
               "Üç yılı dolduran çalışanların izni 24 iş gününe çıkar. Kullanılmayan izinin en fazla 5 günü bir sonraki yıla devredilebilir.",
    "izin-02": "Doğum günü izni: Her çalışan doğum gününün bulunduğu hafta içinde 1 gün ücretli izin kullanabilir. "
               "Bu izin yıllık izinden düşülmez.",
    "uzaktan-01": "Çalışanlar haftada en fazla 3 gün uzaktan çalışabilir. Salı günleri tüm ekiplerin ofiste bulunması zorunludur. "
                  "Uzaktan çalışma günleri en geç bir önceki cuma gününe kadar İK portalına girilmelidir.",
    "egitim-01": "Her çalışanın yıllık kişisel gelişim bütçesi 15.000 TL'dir. Bu bütçe kurs, sertifika sınavı ve konferans "
                 "katılımı için kullanılabilir. Harcama öncesinde yönetici onayı gerekir.",
    "yemek-01": "Ofiste çalışılan her gün için yemek kartına günlük 450 TL yüklenir. Uzaktan çalışılan günlerde yemek kartına yükleme yapılmaz.",
    "vpn-01": "Şirket ağına dışarıdan bağlanmak için 'KYVPN' uygulaması kullanılır. İlk kurulumda BT ekibinden alınan "
              "QR kod ile iki faktörlü doğrulama etkinleştirilir. VPN şifresi 90 günde bir değiştirilmelidir.",
    "donanim-01": "Yeni başlayan yazılım geliştiricilere 32 GB RAM'li dizüstü bilgisayar ve harici monitör verilir. "
                  "Donanım yenileme süresi 3 yıldır. Arızalı cihazlar için destek@kuzeyyildizi.example adresine yazılmalıdır.",
    "mesai-01": "Esnek mesai uygulanır: Çekirdek çalışma saatleri 10:00 ile 16:00 arasıdır. Günlük toplam çalışma süresi 8 saattir.",
    "performans-01": "Performans değerlendirmeleri yılda iki kez, haziran ve aralık aylarında yapılır. "
                     "Değerlendirme sonuçlarına göre maaş artışları ocak ayında uygulanır.",
    "saglik-01": "Tüm çalışanlar ve birinci derece aileleri özel sağlık sigortası kapsamındadır. "
                 "Yıllık check-up hizmeti sigorta tarafından ücretsiz karşılanır.",
}
print(f"Bilgi tabanı: {len(bilgi_tabani)} doküman")

### Retriever: TF-IDF (karakter n-gram)

Türkçede `iznim`, `izni`, `izinler` gibi çekimli kelimeler kelime seviyesinde ayrı terimlerdir. **Karakter n-gram** analizörü ortak kökleri yakalar. Üretim sistemlerinde genellikle yoğun embedding modelleri (ör. `bge-m3`, `multilingual-e5`) + vektör veritabanı (FAISS, Chroma, Qdrant) veya BM25 + embedding **hibrit** arama kullanılır; prensip aynıdır.

In [ ]:
class TfidfRetriever:
    def __init__(self, dokumanlar):
        self.idler = list(dokumanlar)
        self.metinler = list(dokumanlar.values())
        self.vektorlestirici = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
        self.matris = self.vektorlestirici.fit_transform([m.lower() for m in self.metinler])

    def getir(self, soru, k=2):
        skorlar = cosine_similarity(self.vektorlestirici.transform([soru.lower()]), self.matris)[0]
        en_iyi = skorlar.argsort()[::-1][:k]
        return [(self.idler[i], self.metinler[i], float(skorlar[i])) for i in en_iyi]

retriever = TfidfRetriever(bilgi_tabani)

for soru in ["Yıllık iznim kaç gün?", "Evden kaç gün çalışabilirim?", "VPN şifresini ne sıklıkla değiştirmeliyim?"]:
    print(f"\n❓ {soru}")
    for doc_id, _, skor in retriever.getir(soru, k=3):
        print(f"   {skor:.3f}  {doc_id}")

### Retriever değerlendirmesi: Hit@k

Doğru dokümanın ilk $k$ sonuç içinde gelip gelmediği. Generator ne kadar iyi olursa olsun, retriever doğru parçayı getiremezse RAG başarısız olur.

In [ ]:
test_sorulari = [
    ("Üç yıldan sonra izin hakkım ne kadar oluyor?", "izin-01"),
    ("Doğum günümde izin alabilir miyim?", "izin-02"),
    ("Hangi gün ofise gelmek zorunlu?", "uzaktan-01"),
    ("Konferansa gitmek için şirket bütçesi var mı?", "egitim-01"),
    ("Yemek kartına ne kadar para yükleniyor?", "yemek-01"),
    ("Şirket ağına evden nasıl bağlanırım?", "vpn-01"),
    ("Bilgisayarım bozuldu kime yazmalıyım?", "donanim-01"),
    ("Sabah kaçta işte olmam lazım?", "mesai-01"),
    ("Maaş zammı ne zaman yapılıyor?", "performans-01"),
    ("Ailem sağlık sigortasına dahil mi?", "saglik-01"),
]

for k in [1, 2, 3]:
    isabet = np.mean([dogru in [d for d, _, _ in retriever.getir(s, k)] for s, dogru in test_sorulari])
    print(f"Hit@{k}: %{isabet * 100:.0f}")

### Generator: küçük bir açık kaynak LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ADI = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ADI).to(cihaz)
if cihaz == "cuda":
    llm = llm.half()
llm.eval()

@torch.no_grad()
def uret(mesajlar, max_yeni_token=200):
    metin = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    girdi = tokenizer(metin, return_tensors="pt").to(cihaz)
    cikti = llm.generate(**girdi, max_new_tokens=max_yeni_token, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(cikti[0, girdi["input_ids"].shape[1]:], skip_special_tokens=True).strip()

### RAG istemi (prompt)

İyi bir RAG istemi üç şey yapar: (1) modeli **sadece bağlamı kullanmaya** zorlar, (2) cevap bağlamda yoksa **"bilmiyorum" demesine izin verir**, (3) **kaynak göstermesini** ister (doğrulanabilirlik).

In [ ]:
SISTEM_RAG = ("Sen Kuzey Yıldızı Yazılım'ın İK asistanısın. Soruları YALNIZCA verilen bağlamdaki bilgilere dayanarak, "
              "kısa ve Türkçe cevapla. Cevabın sonunda kullandığın kaynağı köşeli parantez içinde yaz, ör. [izin-01]. "
              "Bağlamda cevap yoksa sadece 'Bu bilgi el kitabında yer almıyor.' de, tahmin yürütme.")

def rag_cevapla(soru, k=2):
    parcalar = retriever.getir(soru, k)
    baglam = "\n\n".join(f"[{doc_id}] {metin}" for doc_id, metin, _ in parcalar)
    mesajlar = [{"role": "system", "content": SISTEM_RAG},
                {"role": "user", "content": f"Bağlam:\n{baglam}\n\nSoru: {soru}"}]
    return uret(mesajlar), [d for d, _, _ in parcalar]

def ragsiz_cevapla(soru):
    return uret([{"role": "system", "content": "Kısa ve Türkçe cevap ver."},
                 {"role": "user", "content": f"Kuzey Yıldızı Yazılım şirketinde: {soru}"}])

In [ ]:
sorular = [
    "Yıllık izin hakkım kaç gün?",
    "Haftada kaç gün uzaktan çalışabilirim ve hangi gün ofiste olmalıyım?",
    "Kişisel gelişim bütçesi ne kadar?",
    "Şirketin hisse senedi opsiyonu programı var mı?",          # bilgi tabanında YOK
]

for soru in sorular:
    cevap, kaynaklar = rag_cevapla(soru)
    print("═" * 100)
    print(f"❓ {soru}\n")
    print(f"🚫 RAG'siz : {ragsiz_cevapla(soru)}\n")
    print(f"✅ RAG'li  : {cevap}")
    print(f"   (getirilen parçalar: {kaynaklar})")

RAG'siz cevaplar akıcı görünse de sayılar **uydurma**dır (model bu şirketi hiç görmedi). RAG'li cevaplar el kitabındaki rakamları kullanır ve kaynak gösterir. Bilgi tabanında olmayan soruda ise doğru davranış "bilmiyorum" demektir.

### RAG'in sınırları ve iyileştirmeler

| Sorun | Çözüm |
|---|---|
| Yanlış parça getirildi | Daha iyi embedding, hibrit arama (BM25 + dense), **reranker** (cross-encoder) |
| Parça çok uzun / bağlamı bölünmüş | Akıllı chunking (başlığa/paragrafa göre), örtüşen (overlap) pencereler |
| Model bağlamı yok sayıyor | Talimatı güçlendir, düşük temperature, daha güçlü model |
| Çok adımlı sorular | Sorgu yeniden yazma, çoklu getirme, agentic RAG |
| Kaynak doküman güvenilmez | Getirilen içerik **talimat değil veridir** → dolaylı prompt enjeksiyonu riski (`09_llm_guvenligi` defteri) |

---
# Bölüm B — RLHF ve DPO ile Hizalama

### LLM eğitim aşamaları

```
1) Ön eğitim (pretraining)         → trilyonlarca token, sonraki token tahmini     → "bilgili ama yönsüz" model
2) SFT (supervised fine-tuning)    → binlerce (istem, iyi cevap) örneği             → talimat takip eden model  = π_ref
3) Tercih optimizasyonu            → (istem, tercih edilen, reddedilen) üçlüleri    → yardımsever, zararsız, dürüst model
```

### RLHF (Ouyang vd., 2022 – InstructGPT)

**Adım 1 – Ödül modeli.** İnsanlara aynı isteme verilmiş iki cevap gösterilir, hangisini tercih ettikleri sorulur. **Bradley–Terry** modeline göre:

$$P(y_w \succ y_l \mid x) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big), \qquad
\mathcal{L}_{RM} = -\log \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big)$$

**Adım 2 – Politika optimizasyonu (PPO).** Model ödülü maksimize ederken referans modelden çok uzaklaşmaması için **KL cezası** alır (yoksa ödül modelinin açıklarını sömürür: *reward hacking*):

$$\max_{\pi_\theta}\; \mathbb{E}_{x,\, y \sim \pi_\theta}\big[r_\phi(x, y)\big] - \beta\, \mathrm{KL}\big(\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big)$$

Bu problemin kapalı form çözümü: $\pi^*(y|x) \propto \pi_{\text{ref}}(y|x)\, \exp\big(r(x,y)/\beta\big)$

### DPO (Rafailov vd., 2023)

Kapalı form çözümü ters çevirirsek ödül, politikanın kendisi cinsinden yazılabilir: $r(x,y) = \beta \log\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} + \text{sabit}$. Bunu Bradley–Terry kaybına koyunca **ödül modeli ve RL döngüsü tamamen ortadan kalkar**:

$$\mathcal{L}_{DPO} = -\log \sigma\left(\beta \left[\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

| | RLHF (PPO) | DPO |
|---|---|---|
| Ödül modeli | Ayrı bir model eğitilir | **Yok** (örtük) |
| Eğitim sırasında örnekleme | Gerekli (on-policy üretim) | Gerekli değil (offline veri) |
| Bellekteki modeller | Politika, referans, ödül, değer (critic) | Politika, referans |
| Kararlılık / ayar | Hassas, çok hiperparametre | Basit, sınıflandırma kaybı gibi |
| $\beta$ | KL cezasının gücü | Aynı rol: referanstan sapma kontrolü |

### Oyuncak problem

Gerçek bir LLM'i burada RLHF ile eğitmek mümkün değil; ama matematik tamamen aynı olduğundan problemi **küçültüyoruz**:
- 30 farklı istem, her istem için 6 aday cevap.
- Her cevabın 3 özelliği var: **yararlılık**, **nezaket**, **uzunluk**.
- İnsanların gizli "gerçek" tercihi: $r^* = 2\cdot\text{yararlılık} + 1\cdot\text{nezaket} - 0.7\cdot\text{uzunluk}$ (yararlı ve kibar olsun, gereksiz uzatmasın).
- SFT modeli ($\pi_{\text{ref}}$) ise kusurlu: **uzun cevapları** fazla seviyor.

In [ ]:
N_ISTEM, N_CEVAP = 30, 6
ozellik_adlari = ["yararlılık", "nezaket", "uzunluk"]
ozellikler = torch.randn(N_ISTEM, N_CEVAP, 3)
gercek_w = torch.tensor([2.0, 1.0, -0.7])
gercek_odul = ozellikler @ gercek_w                                   # (istem, cevap)

ref_logit = 1.2 * ozellikler[..., 2] + 0.3 * ozellikler[..., 0] + 0.3 * torch.randn(N_ISTEM, N_CEVAP)
ref_logp = F.log_softmax(ref_logit, dim=-1)
ref_olasilik = ref_logp.exp()

def beklenen_odul(logp):
    return (logp.exp() * gercek_odul).sum(-1).mean().item()

def kl_ref(logp):
    return (logp.exp() * (logp - ref_logp)).sum(-1).mean().item()

print(f"SFT (π_ref) beklenen gerçek ödül: {beklenen_odul(ref_logp):.3f}")
print(f"En iyi cevabı her zaman seçen ideal politika: {gercek_odul.max(-1).values.mean():.3f}")

### Tercih verisi toplamak (insan etiketleyicileri simüle ediyoruz)

Her örnekte bir istem seçilir, $\pi_{\text{ref}}$'ten iki farklı cevap örneklenir, "insan" Bradley–Terry olasılığıyla birini tercih eder (yani bazen gürültülü/tutarsız etiket verir, tıpkı gerçek insanlar gibi).

In [ ]:
def tercih_verisi_uret(n):
    istemler, kazananlar, kaybedenler = [], [], []
    for _ in range(n):
        x = random.randrange(N_ISTEM)
        y1, y2 = torch.multinomial(ref_olasilik[x], 2, replacement=False).tolist()
        p_y1 = torch.sigmoid(gercek_odul[x, y1] - gercek_odul[x, y2]).item()
        kazanan, kaybeden = (y1, y2) if random.random() < p_y1 else (y2, y1)
        istemler.append(x); kazananlar.append(kazanan); kaybedenler.append(kaybeden)
    return torch.tensor(istemler), torch.tensor(kazananlar), torch.tensor(kaybedenler)

x_egt, yw_egt, yl_egt = tercih_verisi_uret(3000)
x_tst, yw_tst, yl_tst = tercih_verisi_uret(1000)
print(f"Eğitim tercih çifti: {len(x_egt)} | Test tercih çifti: {len(x_tst)}")
print(f"Örnek: istem {x_egt[0].item()} → cevap {yw_egt[0].item()} ≻ cevap {yl_egt[0].item()}")

## B.1) RLHF Adım 1 — Ödül modelini eğitmek

In [ ]:
odul_w = torch.zeros(3, requires_grad=True)
opt = torch.optim.Adam([odul_w], lr=0.05)
rm_kayiplar = []
for _ in range(300):
    r_w = ozellikler[x_egt, yw_egt] @ odul_w
    r_l = ozellikler[x_egt, yl_egt] @ odul_w
    kayip = -F.logsigmoid(r_w - r_l).mean()                         # Bradley–Terry kaybı
    opt.zero_grad(); kayip.backward(); opt.step()
    rm_kayiplar.append(kayip.item())

with torch.no_grad():
    rm_dogruluk = ((ozellikler[x_tst, yw_tst] @ odul_w) > (ozellikler[x_tst, yl_tst] @ odul_w)).float().mean()
    ust_sinir = (gercek_odul[x_tst, yw_tst] > gercek_odul[x_tst, yl_tst]).float().mean()

print("Öğrenilen ödül ağırlıkları:", dict(zip(ozellik_adlari, odul_w.detach().numpy().round(2))))
print("Gerçek ödül ağırlıkları   :", dict(zip(ozellik_adlari, gercek_w.numpy())))
print(f"\nTest tercih doğruluğu: %{rm_dogruluk * 100:.1f}  (gürültülü etiketler nedeniyle üst sınır ≈ %{ust_sinir * 100:.1f})")

## B.2) RLHF Adım 2 — KL kısıtlı politika

Gerçek LLM'de cevap uzayı astronomik büyüklükte olduğu için PPO ile örnekleyerek optimize edilir. Bizim oyuncak problemimizde cevap uzayı küçük, bu yüzden **optimal çözümü doğrudan** hesaplayabiliriz:

$$\pi^*_{\text{RLHF}}(y|x) \propto \pi_{\text{ref}}(y|x)\, \exp\big(\hat r(x,y)/\beta\big)$$

In [ ]:
with torch.no_grad():
    tahmini_odul = ozellikler @ odul_w

def rlhf_politikasi(beta):
    return F.log_softmax(ref_logp + tahmini_odul / beta, dim=-1)

## B.3) DPO — Ödül modeli olmadan, doğrudan tercih verisinden

In [ ]:
def dpo_egit(beta, adim=1500, lr=0.05):
    politika_logit = ref_logit.clone().requires_grad_(True)          # π_θ, SFT modelinden başlar
    opt = torch.optim.Adam([politika_logit], lr=lr)
    for _ in range(adim):
        logp = F.log_softmax(politika_logit, dim=-1)
        oran_w = logp[x_egt, yw_egt] - ref_logp[x_egt, yw_egt]      # log π_θ(y_w|x) − log π_ref(y_w|x)
        oran_l = logp[x_egt, yl_egt] - ref_logp[x_egt, yl_egt]
        kayip = -F.logsigmoid(beta * (oran_w - oran_l)).mean()
        opt.zero_grad(); kayip.backward(); opt.step()
    return F.log_softmax(politika_logit.detach(), dim=-1)

dpo_logp = dpo_egit(beta=0.5)
rlhf_logp = rlhf_politikasi(beta=0.5)

print(f"{'Politika':22s} {'Beklenen gerçek ödül':>22s} {'KL(π || π_ref)':>16s}")
for isim, logp in [("SFT (π_ref)", ref_logp), ("RLHF (β=0.5)", rlhf_logp), ("DPO  (β=0.5)", dpo_logp)]:
    print(f"{isim:22s} {beklenen_odul(logp):22.3f} {kl_ref(logp):16.3f}")

In [ ]:
fig, eksenler = plt.subplots(1, 3, figsize=(18, 4.2), sharey=True)
for ax, x in zip(eksenler, [0, 1, 2]):
    sira = gercek_odul[x].argsort(descending=True)
    konum = np.arange(N_CEVAP)
    ax.bar(konum - 0.27, ref_olasilik[x, sira], 0.27, label="SFT (π_ref)")
    ax.bar(konum, rlhf_logp[x, sira].exp(), 0.27, label="RLHF")
    ax.bar(konum + 0.27, dpo_logp[x, sira].exp(), 0.27, label="DPO")
    ax.set_xticks(konum)
    ax.set_xticklabels([f"r*={gercek_odul[x, i]:.1f}\nuz={ozellikler[x, i, 2]:.1f}" for i in sira], fontsize=8)
    ax.set_title(f"İstem {x}: cevaplar gerçek ödüle göre sıralı (sol = en iyi)")
eksenler[0].set_ylabel("olasılık"); eksenler[0].legend()
plt.tight_layout(); plt.show()

SFT modeli **uzun** cevaplara yüksek olasılık veriyordu. RLHF ve DPO, olasılık kütlesini insanların gerçekten tercih ettiği (yararlı, kibar, gereksiz uzun olmayan) cevaplara kaydırdı.

### $\beta$'nın rolü: ödül ↔ referanstan sapma dengesi

Küçük $\beta$ → referanstan çok uzaklaşabilir (daha agresif optimizasyon, aşırı uyma ve *reward hacking* riski). Büyük $\beta$ → referansa yakın kalır.

In [ ]:
satirlar = []
for beta in [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]:
    r_logp, d_logp = rlhf_politikasi(beta), dpo_egit(beta, adim=800)
    satirlar.append({"β": beta,
                     "RLHF ödül": beklenen_odul(r_logp), "RLHF KL": kl_ref(r_logp),
                     "DPO ödül": beklenen_odul(d_logp), "DPO KL": kl_ref(d_logp)})
beta_tablo = pd.DataFrame(satirlar)

fig, eksenler = plt.subplots(1, 2, figsize=(14, 4.2))
for yontem, stil in [("RLHF", "o-"), ("DPO", "s--")]:
    eksenler[0].semilogx(beta_tablo["β"], beta_tablo[f"{yontem} ödül"], stil, label=yontem)
    eksenler[1].semilogx(beta_tablo["β"], beta_tablo[f"{yontem} KL"], stil, label=yontem)
eksenler[0].axhline(beklenen_odul(ref_logp), color="gray", ls=":", label="SFT")
eksenler[0].set_title("Beklenen gerçek ödül"); eksenler[0].set_xlabel("β")
eksenler[1].set_title("KL(π ‖ π_ref): referanstan sapma"); eksenler[1].set_xlabel("β")
for ax in eksenler:
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
beta_tablo.round(3)

### Gerçek bir LLM'de DPO: Hugging Face TRL

```python
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

# Her satır: {"prompt": ..., "chosen": ..., "rejected": ...}
veri = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:2000]")

ayar = DPOConfig(output_dir="qwen-dpo", beta=0.1, per_device_train_batch_size=2,
                 learning_rate=5e-7, num_train_epochs=1)
trainer = DPOTrainer(model=model, args=ayar, train_dataset=veri, processing_class=tokenizer)
trainer.train()
```

Referans model belirtilmezse TRL, eğitim başındaki modelin donmuş bir kopyasını $\pi_{\text{ref}}$ olarak kullanır. LoRA ile birleştirilirse (`peft_config=...`) tek bir T4 GPU'da bile çalıştırılabilir.

---
## Özet

| Teknik | Problem | Fikir |
|---|---|---|
| **RAG** | Halüsinasyon, güncel olmayan / kuruma özel bilgi | Cevaplamadan önce bilgi tabanından ilgili parçaları getir, bağlama koy, kaynak göster |
| **Ödül modeli** | İnsan tercihini sayıya çevirmek | Bradley–Terry: $-\log\sigma(r_w - r_l)$ |
| **RLHF (PPO)** | Modeli tercihlere göre yönlendirmek | Ödülü maksimize et − $\beta\cdot$KL(π‖π_ref) |
| **DPO** | RLHF'nin karmaşıklığı ve kararsızlığı | Ödül = $\beta\log\frac{\pi}{\pi_{\text{ref}}}$ → tercih çiftleri üzerinde doğrudan sınıflandırma kaybı |
| **$\beta$** | Reward hacking, aşırı sapma | Referans modele ne kadar bağlı kalınacağını belirler |